In [ ]:
import torch.nn as nn
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models
from torch.optim import Adam
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torchvision
from tensorboardX import SummaryWriter
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt

In [ ]:
transforms_train = transforms.Compose([
    #transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transforms_test = transforms.Compose([
    #transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
full_datasets = datasets.ImageFolder(root='../datasets/data')
class_names = full_datasets.classes
class_names

In [ ]:
label_counts = Counter(full_datasets.targets)

print("Class distribution in full_datasets:")
print(label_counts)

# 클래스 인덱스 순서대로 정렬하여 리스트 생성
sorted_counts = [label_counts[i] for i in sorted(label_counts.keys())]

# 각 클래스 데이터 수의 역수 계산
class_weights = 1.0 / torch.tensor(sorted_counts, dtype=torch.float)

# 전체 합이 1이 되도록 정규화
class_weights /= class_weights.sum()

print("계산된 클래스 가중치:", class_weights)

In [ ]:
# 데이터 크기와 target 저장 
total_size = len(full_datasets)
targets = full_datasets.targets

train_indices, test_indices = train_test_split(
    np.arange(total_size),
    test_size=0.2,
    stratify=targets,
    random_state=42   
)

train_dataset = torch.utils.data.Subset(full_datasets, train_indices)
test_dataset = torch.utils.data.Subset(full_datasets, test_indices)

print("\n데이터셋 분할 완료.")
print(f"총 데이터 수: {total_size}")
print(f"학습용 데이터 수: {len(train_dataset)}")
print(f"검증용 데이터 수: {len(test_dataset)}")

In [ ]:
train_dataset.dataset.trnsform = transforms_train
test_dataset.dataset.transform = transforms_test

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
## 이미지 시각화를 함수로 정의

def imshow(img, title):
    img = img.numpy().transpose((1,2,0))

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    img = std * img + mean
    img = np.clip(img, 0, 1)

    plt.imshow(img)
    plt.title(title)
    plt.axis("off")
    plt.show()
    

In [ ]:
images, labels = next(iter(train_loader))
images_data = torchvision.utils.make_grid(images)

imshow(images_data, labels)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
weights1 = torch.load("model/lr1e4_512best_efficientB0_model_pretrained_weights827.pth")
test_model = models.efficientnet_b0(pretrained=None)
test_model.classifier[1] = nn.Linear(in_features=1280, out_features=13, bias=True)
test_model.load_state_dict(weights1)
test_model.to(device)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import classification_report

y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = test_model(images)
        probs = torch.softmax(logits, dim=1)

        preds = torch.argmax(logits, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

        

In [ ]:
# 스코어 계산
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')
accuracy = (np.array(y_true) == np.array(y_pred)).mean()

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print(classification_report(y_true, y_pred, target_names=class_names))

#### 한 배치만 꺼내서 이미지 시각화와 실제레이블, 예측 레이블, 예측 확률 보기

In [ ]:
# test_loader 에서 첫번째 배치 가져오기
images, labels = next(iter(test_loader))

# 디바이스로 데이터 이동
images, labels = images.to(device), labels.to(device)

#모델 예측
with torch.no_grad():
    logits = test_model(images)

    probs = torch.softmax(logits, dim=1)
    preds = torch.argmax(probs, dim=1)

    #예측된 레이블의 확률
    predicted_probs = torch.max(probs, dim=1).values

#Numpy 배열로 변환
images = images.cpu().numpy()
labels = labels.cpu().numpy()
preds = preds.cpu().numpy()
predicted_probs = predicted_probs.cpu().numpy()

#역정규화 함수
NORM_MEAN = [0.486, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

def denormalize_image(tensor_image, mean=NORM_MEAN, std=NORM_STD):
    np_image = tensor_image.transpose(1, 2, 0) # (C, H, W) -> (H, W, C)
    denorm_image = (np_image * np.array(std)) + np.array(mean)
    return np.clip(denorm_image, 0, 1)

# 첫 번째 배치 내 각 이미지에 대한 예측 출력
print(f"총 32개의 이미지 정보를 출력합니다.")
for i in range(len(images)):
    image_data = images[i]
    true_label = labels[i]
    pred_label = preds[i]
    pred_prob = predicted_probs[i]

    # 이미지 역정규화
    denoormalize_image = denormalize_image(image_data)

    #시각화
    plt.figure(figsize=(4,4))
    plt.imshow(denoormalize_image)
    plt.title(f"True Label: {class_names[true_label]}\nPredicted: {class_names[pred_label]} (Prob: {pred_prob:.2f})")
    plt.axis("off")

    # ✅ plt.show() 전에 저장
    # 파일명에 인덱스를 추가하여 각 이미지가 다른 이름으로 저장되도록 수정
    save_path = f"C:/Users/user/Downloads/pred_{i}.png"
    plt.savefig(save_path)
    
    plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import koreanize_matplotlib

cm = confusion_matrix(y_true, y_pred)

# 히트맵 시각화
plt.figure(figsize=(12,10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix Heatmap", fontsize=16)
plt.ylabel("True Label", fontsize=14)
plt.xlabel("Predicted Label", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
weights1 = torch.load("model/512best_efficientB0_model_weights827.pth")
test_model = models.efficientnet_b0(pretrained=None)
test_model.classifier[1] = nn.Linear(in_features=1280, out_features=13, bias=True)
test_model.load_state_dict(weights1)
test_model.to(device)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import classification_report

y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = test_model(images)
        probs = torch.softmax(logits, dim=1)

        preds = torch.argmax(logits, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

In [ ]:
# 스코어 계산
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')
accuracy = (np.array(y_true) == np.array(y_pred)).mean()

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import koreanize_matplotlib

cm = confusion_matrix(y_true, y_pred)

# 히트맵 시각화
plt.figure(figsize=(12,10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix Heatmap", fontsize=16)
plt.ylabel("True Label", fontsize=14)
plt.xlabel("Predicted Label", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()